In [ ]:
import os
import random
import sys
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from scipy.interpolate import griddata
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import tqdm

# Ensure repo parent is on sys.path so absolute imports like 'HydroSynth' work.
_repo_root = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
_repo_root = os.path.normpath(_repo_root)
_repo_parent = os.path.dirname(_repo_root)
if _repo_parent not in sys.path:
    sys.path.insert(0, _repo_parent)

from HydroSynth import config
try:
    from HydroSynth.FNO import model1
except Exception:
    import model1  # fallback for direct script execution from HydroSynth/FNO

from train import *

In [ ]:
seed = int(config.modelconfig.get("seed", 42))
set_seed(seed)

device = config.modelconfig["device"]
data = prepare_data()
train_loader, val_loader, test_loader = build_dataloaders(data, device=device)

model = build_model(data, device=device)

weight_name = "epoch_500.pt"
load_dir = r"D:\workplace\conv_data\weight_t0\run_20260316_184706"
ckpt_path = os.path.join(load_dir, weight_name)
ckpt = torch.load(ckpt_path, map_location=device)
if isinstance(ckpt, dict):
    state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
else:
    state_dict = ckpt.state_dict()
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(
    f"Loaded checkpoint: {ckpt_path}\n"
    f"Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}"
)

lr = float(config.modelconfig.get("lr", 3e-4))
weight_decay = float(config.modelconfig.get("weight_decay", 1e-4))
epochs = int(config.modelconfig.get("epoch", 80))
grad_accum = int(config.modelconfig.get("grad_accum", 4))
grad_clip = float(config.modelconfig.get("grad_clip", 1.0))
save_every = int(config.modelconfig.get("save_every", 5))
patience = int(config.modelconfig.get("patience", 12))
min_delta = float(config.modelconfig.get("early_stop_min_delta", 1e-4))

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


print(
    "Train setup:",
    {
        "device": str(device),
        "batch_size": train_loader.batch_size,
        "epochs": epochs,
        "grad_accum": grad_accum,
        "lr": lr,
        "weight_decay": weight_decay,
    },
)

baseline_val = evaluate_baseline(val_loader, device=device)
baseline_test = evaluate_baseline(test_loader, device=device)
print(format_metric_line("Baseline VAL RMSE", baseline_val["rmse"]))
print(format_metric_line("Baseline TEST RMSE", baseline_test["rmse"]))

best_val_loss = float("inf")
best_epoch = -1
stale_epochs = 0

global_step = 0

model.train()
train_losses: List[float] = []
train_state = init_metric_state()
skipped_batches = 0

optimizer.zero_grad(set_to_none=True)
accum_counter = 0
pbar = tqdm.tqdm(train_loader)
for step, (cond, ec_base, target, obs_mask, sst_pcs) in enumerate(pbar):
    cond = cond.to(device, non_blocking=True)
    ec_base = ec_base.to(device, non_blocking=True)
    target = target.to(device, non_blocking=True)
    obs_mask = obs_mask.to(device, non_blocking=True)
    sst_pcs = sst_pcs.to(device, non_blocking=True)

    pred = model(cond, ec_base=ec_base, sst_pcs=sst_pcs)
    loss, huber, mse_res, valid_count = compute_loss(pred, target, ec_base, obs_mask)

    if valid_count == 0:
        skipped_batches += 1
        continue

    if (not torch.isfinite(pred).all().item()) or (not torch.isfinite(loss).item()):
        raise FloatingPointError(
            "Non-finite values detected in training. "
            "Check input normalization."
        )

    scaled_loss = loss / max(1, grad_accum)
    scaled_loss.backward()
    accum_counter += 1

    do_step = accum_counter >= max(1, grad_accum)
    if do_step:
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        accum_counter = 0

    train_losses.append(float(loss.item()))
    pred_det = pred.detach()
    target_det = target[:, :, 0].detach()
    mask_det = (obs_mask[:, :, 0] > 0.5).detach()
    update_metrics(train_state, pred_det, target_det, mask_det)

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        huber=f"{huber.item():.4f}",
        mse=f"{mse_res.item():.4f}",
        skipped=skipped_batches,
    )


if accum_counter > 0:
    if grad_clip > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

train_metrics = finalize_metrics(train_state)
train_loss = float(np.mean(train_losses)) if train_losses else float("inf")

model.eval()
val_losses: List[float] = []
val_state = init_metric_state()
with torch.no_grad():
    for cond, ec_base, target, obs_mask, sst_pcs in val_loader:
        cond = cond.to(device, non_blocking=True)
        ec_base = ec_base.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        obs_mask = obs_mask.to(device, non_blocking=True)
        sst_pcs = sst_pcs.to(device, non_blocking=True)

        pred = model(cond, ec_base=ec_base, sst_pcs=sst_pcs)
        loss, _, _, valid_count = compute_loss(pred, target, ec_base, obs_mask)

        if valid_count == 0:
            continue
        if (not torch.isfinite(pred).all().item()) or (not torch.isfinite(loss).item()):
            raise FloatingPointError(
                "Non-finite values detected in validation. "
                "Check input normalization."
            )

        val_losses.append(float(loss.item()))
        update_metrics(val_state, pred, target[:, :, 0], obs_mask[:, :, 0] > 0.5)

val_metrics = finalize_metrics(val_state)
val_loss = float(np.mean(val_losses)) if val_losses else float("inf")


print(format_metric_line("VAL RMSE", val_metrics["rmse"]))
print(format_metric_line("VAL ACC", val_metrics["acc"]))
skill = baseline_val["rmse"] - val_metrics["rmse"]
acc_diff = val_metrics["acc"] - baseline_val["acc"]
print(format_metric_line("VAL RMSE Skill (baseline-model)", skill)) # skill > 0 means model is better than baseline
print(format_metric_line("VAL ACC Skill (baseline-model)", acc_diff)) # acc_diff > 0 means model is better than baseline



model.eval()
test_losses: List[float] = []
test_state = init_metric_state()
with torch.no_grad():
    for cond, ec_base, target, obs_mask, sst_pcs in test_loader:
        cond = cond.to(device, non_blocking=True)
        ec_base = ec_base.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        obs_mask = obs_mask.to(device, non_blocking=True)
        sst_pcs = sst_pcs.to(device, non_blocking=True)

        pred = model(cond, ec_base=ec_base, sst_pcs=sst_pcs)
        loss, _, _, valid_count = compute_loss(pred, target, ec_base, obs_mask)
        if valid_count == 0:
            continue
        test_losses.append(float(loss.item()))
        update_metrics(test_state, pred, target[:, :, 0], obs_mask[:, :, 0] > 0.5)

test_metrics = finalize_metrics(test_state)
test_loss = float(np.mean(test_losses)) if test_losses else float("inf")
print(f"Final TEST loss={test_loss:.5f}")
print(format_metric_line("TEST RMSE", test_metrics["rmse"]))
print(format_metric_line("TEST ACC", test_metrics["acc"]))
test_skill = baseline_test["rmse"] - test_metrics["rmse"]
print(format_metric_line("TEST RMSE Skill (baseline-model)", test_skill))
